# M3Net + Ada2I (AFW/AMW) + PMR — Emotion Recognition in Conversation

Trains the CVPR 2023 M3Net architecture on pre-extracted RoBERTa / acoustic /
visual features (IEMOCAP or MELD), with three optional modality-balancing
mechanisms layered on top:

| Module | Origin | What it does |
|---|---|---|
| **AFW** | Ada2I (ACM MM 2024) | Tensor-attention reweighting of the per-modality post-graph features, plus an L1 consistency loss |
| **AMW** | Ada2I | Shrinks the gradient reaching a *dominant* modality's encoder, in proportion to how far ahead it is |
| **PCE** | PMR (CVPR 2023) | *Actively accelerates* the slow-learning modalities with a prototypical cross-entropy loss whose gradient bypasses the fusion stack entirely |
| **PER** | PMR | Maximizes the entropy of the *dominant* modality's prototype distribution during early epochs, delaying its premature convergence |

AMW only brakes the leader; PCE additionally pushes the laggards along a
direction that the dominant modality cannot disturb. In M3Net that distinction
matters more than in PMR's original setting, because the hypergraph
convolutions mix modalities *before* the classifier.

**Section 3** below selects the dataset and the method configuration; **Section 5**
runs a single configuration, the 9-row method ablation, or the modality-subset
ablation. **Section 6** reads back the per-epoch diagnostics — including the
per-modality prototype probe, which is the evidence that the weak modalities
actually learned more rather than the fused head merely re-weighting them.
**Section 7** lists the saved checkpoints and reloads the best one to confirm it
is usable.

Each run saves its best-scoring model to `checkpoints/`, together with the
optimizer state, the epoch and score, the seed, every argument it was launched
with, and the PMR prototype buffers — so a checkpoint can be reloaded and scored
later without remembering how it was trained.


## 1. Environment check

In [67]:
import os
import sys
import platform
from pathlib import Path

import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("Kaggle input exists:", Path("/kaggle/input").exists())

assert torch.cuda.is_available(), (
    "No CUDA GPU was found. In Notebook options, set Accelerator to GPU, "
    "save the setting, and restart the session."
)

print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA runtime: 12.8
Kaggle input exists: True
GPU: Tesla T4


In [68]:
import importlib.util
import subprocess
import sys

def module_missing(name):
    return importlib.util.find_spec(name) is None

if module_missing("torch_geometric"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "torch-geometric"
    ])

if module_missing("torch_scatter"):
    torch_version = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else (
        "cu" + torch.version.cuda.replace(".", "")
    )
    wheel_page = (
        f"https://data.pyg.org/whl/"
        f"torch-{torch_version}+{cuda_tag}.html"
    )
    print("Using PyG wheel page:", wheel_page)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch-scatter", "-f", wheel_page
    ])

if module_missing("ipdb"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "ipdb"
    ])

import torch_geometric
import torch_scatter

print("torch-geometric:", torch_geometric.__version__)
print("torch-scatter: imported successfully")

torch-geometric: 2.8.0.post1
torch-scatter: imported successfully


## 2. Stage the code and features

In [ ]:
import shutil
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_REPO = Path("/kaggle/working/M3NET")

# Excludes any "PMR" subfolder: the parent M3NET repo keeps PMR/ around as a
# reference snapshot (docs plus a pre-integration copy of the source), and if
# the whole repo is uploaded as one Kaggle dataset that subfolder would
# otherwise also match -- since it independently has train.py/model.py/
# model_hyper.py/dataloader.py -- and trip the "more than one" assert below.
repo_candidates = sorted({
    path.parent
    for path in INPUT_ROOT.rglob("train.py")
    if (path.parent / "model.py").exists()
    and (path.parent / "model_hyper.py").exists()
    and (path.parent / "dataloader.py").exists()
    and "PMR" not in path.parent.parts
})

assert repo_candidates, (
    "Could not find the M3Net source under /kaggle/input. "
    "Attach the m3net-code dataset and rerun this cell."
)

assert len(repo_candidates) == 1, (
    "More than one M3Net source directory was found. Set CODE_SOURCE "
    f"manually from: {repo_candidates}"
)

CODE_SOURCE = repo_candidates[0]
print("Code source:", CODE_SOURCE)

if WORK_REPO.exists():
    shutil.rmtree(WORK_REPO)

shutil.copytree(CODE_SOURCE, WORK_REPO)
os.chdir(WORK_REPO)

print("Writable repository:", Path.cwd())
python_files = sorted(path.name for path in WORK_REPO.glob("*.py"))
print("Python files:", python_files)

for required in ["afw.py", "pmr.py", "model.py", "model_hyper.py", "train.py"]:
    assert required in python_files, (
        "{} is missing from the attached code dataset. Re-upload the M3Net "
        "source so it includes the PMR integration.".format(required)
    )
print("AFW + PMR sources present.")

In [70]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_REPO = Path("/kaggle/working/M3NET")

def find_one(filename):
    matches = list(INPUT_ROOT.rglob(filename))
    assert matches, (
        f"Could not find {filename} under /kaggle/input. "
        "Check that the feature dataset is attached."
    )
    assert len(matches) == 1, (
        f"Found multiple copies of {filename}: {matches}. "
        "Choose the intended feature dataset manually."
    )
    return matches[0]

iemocap_main = find_one("IEMOCAP_features.pkl")
iemocap_roberta = find_one("iemocap_features_roberta.pkl")
assert iemocap_main.parent == iemocap_roberta.parent

meld_main = find_one("MELD_features_raw1.pkl")
meld_roberta = find_one("meld_features_roberta.pkl")
assert meld_main.parent == meld_roberta.parent

feature_sources = {
    "IEMOCAP_features": iemocap_main.parent,
    "MELD_features": meld_main.parent,
}

for folder_name, source in feature_sources.items():
    destination = WORK_REPO / folder_name
    if destination.is_symlink():
        destination.unlink()
    elif destination.exists():
        raise RuntimeError(
            f"{destination} already exists and is not a symlink. "
            "Inspect it before replacing it."
        )
    destination.symlink_to(source, target_is_directory=True)
    print(destination, "->", source)

assert (WORK_REPO / "IEMOCAP_features/IEMOCAP_features.pkl").is_file()
assert (WORK_REPO / "IEMOCAP_features/iemocap_features_roberta.pkl").is_file()
assert (WORK_REPO / "MELD_features/MELD_features_raw1.pkl").is_file()
assert (WORK_REPO / "MELD_features/meld_features_roberta.pkl").is_file()

print("All four required feature files are visible.")

/kaggle/working/M3NET/IEMOCAP_features -> /kaggle/input/datasets/shufanshahi/m3net-features/IEMOCAP_features
/kaggle/working/M3NET/MELD_features -> /kaggle/input/datasets/shufanshahi/m3net-features/MELD_features
All four required feature files are visible.


In [71]:
import os
import sys
from pathlib import Path

os.chdir("/kaggle/working/M3NET")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from dataloader import IEMOCAPDataset, MELDDataset
from model import Model

print("M3Net imports succeeded.")

M3Net imports succeeded.


## 3. Configuration

In [ ]:
# ============================ 3. Configuration ============================

CONFIGS = {
    "IEMOCAP": {
        "seed": 1475,
        "base_model": "GRU",
        "dropout": 0.5,
        "lr": 1e-4,
        "batch_size": 16,
        "epochs": 80,
        "graph_type": "hyper",
        "graph_construct": "direct",
        "multi_modal": True,
        "mm_fusion_mthd": "concat_DHT",
        "modals": "avl",
        "norm": "BN",
        "num_L": 3,
        "num_K": 4,
    },
    "MELD": {
        "seed": 67137,
        "base_model": "GRU",
        "dropout": 0.4,
        "lr": 1e-4,
        "batch_size": 16,
        "epochs": 15,
        "graph_type": "hyper",
        "graph_construct": "direct",
        "multi_modal": True,
        "mm_fusion_mthd": "concat_DHT",
        "modals": "avl",
        "norm": "BN",
        "num_L": 3,
        "num_K": 3,
    },
}

# Change only this line to switch datasets.
DATASET = "IEMOCAP"  # "IEMOCAP" or "MELD"

# --------------------------------------------------------------- what to run
#   "single"   -- one run with the METHOD settings below
#   "method"   -- the 9-row AFW/AMW/PCE/PER ablation matrix
#   "modality" -- the same METHOD settings across avl / al / vl / av
RUN_MODE = "single"

# ------------------------------------------------- Ada2I modules (AFW / AMW)
AFW = {
    "enabled": True,
    "rank": 8,
    "beta": 0.7,
    "nlayers": 2,
    "dropout": 0.1,
}

AMW = {
    "enabled": True,
    "alpha": 0.5,
    "mu": 1,            # >0 also boosts the weakest modality's gradient
    "start_epoch": 0,
    "end_epoch": 1000000000,
    # "classifier" = Ada2I's smax_fc column slicing (default);
    # "prototype"  = drive AMW with PMR's fusion-independent rho_m instead.
    "ratio_source": "classifier",
}

# ------------------------------------------------------------------- PMR
# Paper-faithful defaults are alpha=1, mu=1e-2, distance="euclid", temp=1.0,
# start_epoch=0, proto_subset=0.1. The deviations below are deliberate and are
# documented in PMR_M3Net_Ada2I_Implementation_Plan.md:
#   * start_epoch=1  -- one vanilla warm-up epoch gives less degenerate centroids
#                       than computing them from a randomly-initialised model.
#   * proto_subset=1.0 -- M3Net's training sets are small and its features are
#                       pre-extracted, so a full no_grad pass costs seconds.
PMR = {
    "enabled": True,
    "tap": "pre_graph",       # pre_graph | post_graph | post_afw
    "alpha": 1.0,             # PCE weight
    "mu": 1e-2,               # PER weight
    "reg_epochs": 5,          # PER active while epoch < this (0 disables PER)
    "start_epoch": 1,
    "proto_momentum": 0.9,
    "proto_subset": 1.0,
    "distance": "euclid",     # euclid | sq_euclid | cosine
    "temp": 1.0,
    "ratio_ema": 0.9,
    # "linear" = the M3Net baseline single nn.Linear for audio/visual.
    # "mlp"    = 2-layer MLP, so PCE has real capacity to reshape those two
    #            encoders. If you use it, ALSO run it with PMR disabled as a
    #            control -- added capacity alone can move the numbers.
    "av_encoder": "linear",
}

# Per-modality nearest-centroid F1/accuracy each epoch. Independent of PMR being
# enabled, so the baseline produces the same curve for comparison. Cheap.
PMR_PROBE = True

# Per-epoch CSV of grad norms, representation norms, AMW ratio, PMR
# s/rho/beta/gamma/PCE/entropy, and the probe metrics.
LOG_DIAGNOSTICS = True
DIAGNOSTICS_PATH = "diagnostics.csv"

# Angle between the PCE and task gradients (replicates PMR Fig. 2c). Costs two
# extra backward passes on every logged step -- leave off for headline runs.
PMR_LOG_ANGLES = False
PMR_ANGLE_EVERY = 50

# ---------------------------------------------------------------- checkpoints
# train.py rewrites the checkpoint every time the test F1 improves, so what
# lands on disk is the epoch whose score is reported at the end of the run.
# Stored alongside the weights: the epoch and its scores, the seed, every
# argument the run was launched with, and the PMR prototype buffers -- so a
# checkpoint can be reloaded and re-scored without remembering the flags.
SAVE_MODEL = True

# Written under /kaggle/working/M3NET/checkpoints/, which Save Version preserves.
# Each run in an ablation gets its own file (see the run cell). Roughly 45 MiB
# per IEMOCAP checkpoint, so a full 9-row method ablation is ~0.4 GB.
CHECKPOINT_DIR = "checkpoints"

# Adam's state roughly triples the file size and train.py has no resume path,
# so it is left out by default.
SAVE_OPTIMIZER = False

cfg = CONFIGS[DATASET]
print("Dataset:      ", DATASET)
print("Run mode:     ", RUN_MODE)
print("AFW enabled:  ", AFW["enabled"])
print("AMW enabled:  ", AMW["enabled"], "(ratio source: {})".format(AMW["ratio_source"]))
print("PMR enabled:  ", PMR["enabled"],
      "(tap: {}, PER for first {} epochs)".format(PMR["tap"], PMR["reg_epochs"]))
print("Save model:   ", SAVE_MODEL, "-> {}/".format(CHECKPOINT_DIR) if SAVE_MODEL else "")
print("Config:       ", cfg)


In [ ]:
if DATASET == "IEMOCAP":
    dataset = IEMOCAPDataset(train=True)
else:
    dataset = MELDDataset(
        "MELD_features/MELD_features_raw1.pkl",
        train=True,
    )

sample = dataset[0]
tensor_names = [
    "roberta1", "roberta2", "roberta3", "roberta4",
    "visual", "audio", "speaker_mask", "utterance_mask", "labels",
]

print("Number of dialogues:", len(dataset))
print("First dialogue ID:", sample[-1])

for name, value in zip(tensor_names, sample[:-1]):
    print(f"{name:16s} {tuple(value.shape)}")

expected = {
    "IEMOCAP": {"text": 1024, "visual": 342, "audio": 1582},
    "MELD": {"text": 1024, "visual": 342, "audio": 300},
}[DATASET]

assert sample[0].shape[-1] == expected["text"]
assert sample[4].shape[-1] == expected["visual"]
assert sample[5].shape[-1] == expected["audio"]

print("Feature dimensions match train.py.")

## 4. Build the training command

In [ ]:
# ====================== 4. Build the training command ======================
import sys


def build_command(dataset, config, epochs=None, modals=None, overrides=None,
                  save_path=None):
    """Assemble the train.py argv for one run.

    `overrides` is a shallow patch applied on top of the AFW / AMW / PMR dicts
    from the configuration cell, e.g. {"afw": {"enabled": False}}. It is what
    the ablation matrix below uses to vary one row at a time without mutating
    the global settings.

    `save_path` is where the best-test-F1 checkpoint goes; the run cell gives
    each configuration its own file so an ablation does not overwrite itself.
    """
    overrides = overrides or {}
    afw = dict(AFW, **overrides.get("afw", {}))
    amw = dict(AMW, **overrides.get("amw", {}))
    pmr = dict(PMR, **overrides.get("pmr", {}))

    run_epochs = config["epochs"] if epochs is None else epochs
    run_modals = config["modals"] if modals is None else modals

    command = [
        sys.executable, "-u", "train.py",
        "--base-model", config["base_model"],
        "--dropout", str(config["dropout"]),
        "--lr", str(config["lr"]),
        "--batch-size", str(config["batch_size"]),
        "--graph_type", config["graph_type"],
        "--epochs", str(run_epochs),
        "--graph_construct", config["graph_construct"],
        "--mm_fusion_mthd", config["mm_fusion_mthd"],
        "--modals", run_modals,
        "--Dataset", dataset,
        "--norm", config["norm"],
        "--num_L", str(config["num_L"]),
        "--num_K", str(config["num_K"]),
    ]

    if config["multi_modal"]:
        command.append("--multi_modal")

    if afw["enabled"]:
        command += [
            "--use_afw",
            "--afw_rank", str(afw["rank"]),
            "--afw_beta", str(afw["beta"]),
            "--afw_nlayers", str(afw["nlayers"]),
            "--afw_dropout", str(afw["dropout"]),
        ]

    if amw["enabled"]:
        command += [
            "--use_amw",
            "--amw_alpha", str(amw["alpha"]),
            "--amw_mu", str(amw["mu"]),
            "--amw_start_epoch", str(amw["start_epoch"]),
            "--amw_end_epoch", str(amw["end_epoch"]),
            "--amw_ratio_source", amw["ratio_source"],
        ]

    if pmr["enabled"]:
        command += [
            "--use_pmr",
            "--pmr_tap", pmr["tap"],
            "--pmr_alpha", str(pmr["alpha"]),
            "--pmr_mu", str(pmr["mu"]),
            "--pmr_reg_epochs", str(pmr["reg_epochs"]),
            "--pmr_start_epoch", str(pmr["start_epoch"]),
            "--pmr_proto_momentum", str(pmr["proto_momentum"]),
            "--pmr_proto_subset", str(pmr["proto_subset"]),
            "--pmr_distance", pmr["distance"],
            "--pmr_temp", str(pmr["temp"]),
            "--pmr_ratio_ema", str(pmr["ratio_ema"]),
        ]

    # The a/v encoder choice is a *model* setting, so it must be passed whether
    # or not PMR's losses are on -- that is exactly how the no-PMR MLP control
    # is expressed.
    command += ["--pmr_av_encoder", pmr["av_encoder"]]

    if PMR_PROBE:
        command.append("--pmr_probe")

    if PMR_LOG_ANGLES and pmr["enabled"]:
        command += ["--pmr_log_angles", "--pmr_angle_every", str(PMR_ANGLE_EVERY)]

    if LOG_DIAGNOSTICS:
        command += ["--log_diagnostics", "--diagnostics_path", DIAGNOSTICS_PATH]

    if SAVE_MODEL:
        command += ["--save_model", "--save_path",
                    save_path or "{}/{}_model.pth.tar".format(CHECKPOINT_DIR, dataset)]
        if SAVE_OPTIMIZER:
            command.append("--save_optimizer")

    return command


print("Command for the current configuration:\n")
print(" ".join(build_command(DATASET, cfg)))


In [ ]:
# ============================== 4b. Smoke test =============================
# Two epochs is the minimum that exercises PMR: prototypes are first computed at
# epoch `PMR["start_epoch"]`, so a 1-epoch run would never touch the PCE/PER path.
import subprocess
import time

smoke_epochs = max(2, PMR["start_epoch"] + 1) if PMR["enabled"] else 1
# Write to a throwaway path so a 2-epoch smoke run cannot overwrite a real one.
smoke_command = build_command(
    DATASET, cfg, epochs=smoke_epochs,
    save_path="{}/smoke_{}.pth.tar".format(CHECKPOINT_DIR, DATASET))

print("Running smoke test ({} epochs):".format(smoke_epochs))
print(" ".join(smoke_command))

started = time.time()
subprocess.run(smoke_command, cwd="/kaggle/working/M3NET", check=True)
print("\nSmoke test completed in {:.1f} minutes.".format((time.time() - started) / 60))


## 5. Run

In [ ]:
# ========================= 5. Run (single / ablation) =======================
import csv
import datetime
import re
import subprocess
import time
from pathlib import Path

WORK_REPO = Path("/kaggle/working/M3NET")

# Section 7.1 of the implementation plan. `per: False` is expressed as
# --pmr_reg_epochs 0 (PER inactive from the first epoch onward); `pce: False`
# simply drops --use_pmr.
METHOD_ROWS = {
    "1_baseline":     dict(afw=False, amw=False, pce=False, per=False),
    "2_afw_amw":      dict(afw=True,  amw=True,  pce=False, per=False),
    "3_pce":          dict(afw=False, amw=False, pce=True,  per=False),
    "4_pmr":          dict(afw=False, amw=False, pce=True,  per=True),
    "5_amw_pce":      dict(afw=False, amw=True,  pce=True,  per=False),
    "6_afw_pmr":      dict(afw=True,  amw=False, pce=True,  per=True),
    "7_all_but_per":  dict(afw=True,  amw=True,  pce=True,  per=False),
    "8_everything":   dict(afw=True,  amw=True,  pce=True,  per=True),
    "9_proto_ratio":  dict(afw=True,  amw=True,  pce=True,  per=True,
                           ratio_source="prototype"),
}

MODALITY_SUBSETS = ["avl", "al", "vl", "av"]


def row_overrides(row):
    return {
        "afw": {"enabled": row["afw"]},
        "amw": {"enabled": row["amw"],
                "ratio_source": row.get("ratio_source", AMW["ratio_source"])},
        "pmr": {"enabled": row["pce"],
                "reg_epochs": PMR["reg_epochs"] if row["per"] else 0},
    }


F1_RE = re.compile(r"^F-Score:\s*([0-9.]+)", re.M)
ACC_RE = re.compile(r"^Accuracy:\s*([0-9.]+)", re.M)


def run_one(label, command, log_path, save_path=None):
    """Run one configuration, streaming its output live -- every epoch line
    included -- to both the notebook and a log file, then scrape the final
    F1/accuracy that train.py prints."""
    print("\n" + "=" * 74)
    print(">>> {}".format(label))
    print(" ".join(command))
    print("=" * 74, flush=True)

    started = time.time()
    lines = []
    with open(log_path, "w") as log_handle:
        proc = subprocess.Popen(
            command, cwd=str(WORK_REPO), stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True)
            log_handle.write(line)
            lines.append(line)
        proc.wait()
    elapsed = (time.time() - started) / 60.0
    stdout = "".join(lines)

    if proc.returncode != 0:
        print("!! run FAILED (exit {}); see {}".format(proc.returncode, log_path))
        return {"run": label, "f1": None, "accuracy": None,
                "minutes": round(elapsed, 1), "status": "failed", "checkpoint": None}

    f1 = F1_RE.search(stdout)
    acc = ACC_RE.search(stdout)

    checkpoint = None
    if save_path is not None:
        candidate = WORK_REPO / save_path
        if candidate.exists():
            checkpoint = save_path
            print("    checkpoint: {} ({:.1f} MiB)".format(
                save_path, candidate.stat().st_size / 1024 ** 2))
        else:
            print("    !! expected a checkpoint at {} but none was written".format(save_path))

    result = {
        "run": label,
        "f1": float(f1.group(1)) if f1 else None,
        "accuracy": float(acc.group(1)) if acc else None,
        "minutes": round(elapsed, 1),
        "status": "ok",
        "checkpoint": checkpoint,
    }
    print("\n--> {}: F1={} acc={} ({:.1f} min)".format(
        label, result["f1"], result["accuracy"], elapsed))
    return result


stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
results = []
overall_started = time.time()

(WORK_REPO / CHECKPOINT_DIR).mkdir(exist_ok=True)


def checkpoint_for(tag):
    """One checkpoint file per configuration, so an ablation does not
    overwrite itself. Paths are relative to WORK_REPO because train.py runs
    with that as its cwd."""
    return "{}/{}_{}.pth.tar".format(CHECKPOINT_DIR, DATASET, tag)


if RUN_MODE == "single":
    save_path = checkpoint_for("single")
    results.append(run_one(
        "{}_single".format(DATASET),
        build_command(DATASET, cfg, save_path=save_path),
        WORK_REPO / "log_{}_single_{}.txt".format(DATASET, stamp),
        save_path=save_path if SAVE_MODEL else None))

elif RUN_MODE == "method":
    for name, row in METHOD_ROWS.items():
        # Each row gets its own diagnostics file so the CSVs do not interleave.
        globals()["DIAGNOSTICS_PATH"] = "diagnostics_{}_{}.csv".format(DATASET, name)
        save_path = checkpoint_for(name)
        results.append(run_one(
            "{}_{}".format(DATASET, name),
            build_command(DATASET, cfg, overrides=row_overrides(row), save_path=save_path),
            WORK_REPO / "log_{}_{}_{}.txt".format(DATASET, name, stamp),
            save_path=save_path if SAVE_MODEL else None))

elif RUN_MODE == "modality":
    for subset in MODALITY_SUBSETS:
        globals()["DIAGNOSTICS_PATH"] = "diagnostics_{}_{}.csv".format(DATASET, subset)
        save_path = checkpoint_for(subset)
        results.append(run_one(
            "{}_{}".format(DATASET, subset),
            build_command(DATASET, cfg, modals=subset, save_path=save_path),
            WORK_REPO / "log_{}_{}_{}.txt".format(DATASET, subset, stamp),
            save_path=save_path if SAVE_MODEL else None))

else:
    raise ValueError("RUN_MODE must be one of: single, method, modality")

summary_path = WORK_REPO / "results_{}_{}_{}.csv".format(DATASET, RUN_MODE, stamp)
with open(summary_path, "w", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["run", "f1", "accuracy", "minutes", "status", "checkpoint"])
    writer.writeheader()
    writer.writerows(results)

print("\n" + "=" * 74)
print("SUMMARY  ({} mode, {}, total {:.2f} h)".format(
    RUN_MODE, DATASET, (time.time() - overall_started) / 3600))
print("=" * 74)
print("{:<28} {:>8} {:>8} {:>8}  {}".format("run", "F1", "acc", "min", "checkpoint"))
for r in results:
    print("{:<28} {:>8} {:>8} {:>8}  {}".format(
        r["run"], r["f1"] if r["f1"] is not None else "-",
        r["accuracy"] if r["accuracy"] is not None else "-", r["minutes"],
        r["checkpoint"] or "-"))

if RUN_MODE == "method" and len(results) > 1:
    base = next((r["f1"] for r in results if r["run"].endswith("2_afw_amw")), None)
    if base is not None:
        print("\nDelta vs. row 2 (+AFW+AMW, the current best). The single-seed noise")
        print("floor on IEMOCAP is ~0.3-0.4 F1, so promote a row to 3 seeds only if")
        print("it clears +0.5.")
        for r in results:
            if r["f1"] is not None:
                print("  {:<26} {:+.2f}".format(r["run"], r["f1"] - base))

print("\nWritten to", summary_path)


## 6. Diagnostics

In [ ]:
# ==================== 6. Diagnostics: did rebalancing happen? ==============
# Fused F1 alone cannot distinguish "the weak modalities actually learned more"
# from "the fused head re-weighted what it already had". The prototype probe can:
# it is a non-parametric nearest-centroid classifier over each modality's own
# representation. This is the plan's evidence bar item 2 (PMR paper Fig. 4a/b),
# and rho_m over epochs is item 3 (Fig. 4c).
import csv
from pathlib import Path

WORK_REPO = Path("/kaggle/working/M3NET")

diag_files = sorted(WORK_REPO.glob("diagnostics*.csv"))
if not diag_files:
    print("No diagnostics CSV found. Set LOG_DIAGNOSTICS = True and re-run.")

for path in diag_files:
    with open(path) as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        continue

    print("\n" + "=" * 78)
    print(path.name)
    print("=" * 78)

    modalities = sorted({r["modality"] for r in rows})
    epochs = sorted({int(r["epoch"]) for r in rows})

    def cell(epoch, modality, column):
        for r in rows:
            if int(r["epoch"]) == epoch and r["modality"] == modality:
                try:
                    value = float(r.get(column, "nan"))
                except ValueError:
                    return None
                return None if value != value else value
        return None

    def table(title, column, fmt="{:>7.3f}"):
        if all(cell(e, m, column) is None for e in epochs for m in modalities):
            return
        print("\n{}  ({})".format(title, column))
        print("  {:>5} ".format("epoch") + "".join("{:>9}".format(m) for m in modalities))
        for e in epochs:
            line = "  {:>5} ".format(e)
            for m in modalities:
                v = cell(e, m, column)
                line += "{:>9}".format("-" if v is None else fmt.format(v))
            print(line)

    # Item 2: did the weak modalities' own representations get better?
    table("Prototype probe, test weighted-F1 per modality", "pmr_probe_f1", "{:>7.2f}")
    # Item 3: is the imbalance shrinking?
    table("PMR imbalance ratio rho (1.0 = weakest; higher = more dominant)", "pmr_rho")
    # Cross-check: PMR's fusion-independent ratio vs AMW's classifier-based one.
    table("AMW classifier ratio (compare against rho above)", "amw_ratio")
    table("PCE weight beta (accelerate)", "pmr_beta")
    table("PER weight gamma (decelerate)", "pmr_gamma")
    table("Prototype confidence s_m (EMA)", "pmr_s_ema")
    table("PCE loss per modality", "pmr_pce")
    table("Encoder gradient norm", "grad_norm")
    # Item 4 (optional, only with --pmr_log_angles): the direction-interference claim.
    table("PCE-vs-task gradient angle, degrees", "pmr_grad_angle", "{:>7.1f}")

    first, last = epochs[0], epochs[-1]
    probe_first = {m: cell(first, m, "pmr_probe_f1") for m in modalities}
    probe_last = {m: cell(last, m, "pmr_probe_f1") for m in modalities}
    if any(v is not None for v in probe_last.values()):
        print("\nProbe F1 change, epoch {} -> {}:".format(first, last))
        for m in modalities:
            a, b = probe_first.get(m), probe_last.get(m)
            if a is not None and b is not None:
                print("  {}: {:+.2f}".format(m, b - a))
        print("\nA gain here on the weak modalities (a, v) is what separates genuine")
        print("rebalancing from PCE merely acting as an encoder regularizer.")


## 7. Checkpoints

In [ ]:
# ======================= 7. Checkpoints: verify and reload =================
# train.py saved the model each time the test F1 improved. This cell lists what
# landed on disk and reloads the best one to prove the file is usable -- a
# checkpoint you have never loaded is a checkpoint you do not have.
import torch
from pathlib import Path

WORK_REPO = Path("/kaggle/working/M3NET")
ckpt_dir = WORK_REPO / CHECKPOINT_DIR

checkpoints = sorted(ckpt_dir.glob("*.pth.tar")) if ckpt_dir.exists() else []

if not checkpoints:
    print("No checkpoints found in {}.".format(ckpt_dir))
    print("Set SAVE_MODEL = True in the configuration cell and re-run section 5.")
else:
    print("{:<44} {:>9}  {:>7}  {:>6}  {}".format(
        "checkpoint", "MiB", "F1", "epoch", "modules"))
    print("-" * 92)
    for path in checkpoints:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        saved_args = payload.get("args", {})
        modules = [name for name, flag in [
            ("AFW", saved_args.get("use_afw")),
            ("AMW", saved_args.get("use_amw")),
            ("PMR", saved_args.get("use_pmr")),
        ] if flag]
        if saved_args.get("use_pmr") and saved_args.get("pmr_reg_epochs", 0) == 0:
            modules = [m if m != "PMR" else "PCE" for m in modules]
        print("{:<44} {:>9.1f}  {:>7}  {:>6}  {}".format(
            str(path.relative_to(WORK_REPO)),
            path.stat().st_size / 1024 ** 2,
            payload.get("test_fscore", "-"),
            payload.get("epoch", "-"),
            "+".join(modules) or "baseline"))

    # ---- reload the highest-scoring checkpoint back into a fresh Model
    best_path = max(checkpoints, key=lambda p: torch.load(
        p, map_location="cpu", weights_only=False).get("test_fscore", -1))
    payload = torch.load(best_path, map_location="cpu", weights_only=False)
    saved_args = payload["args"]

    print("\nReloading best checkpoint:", best_path.relative_to(WORK_REPO))
    print("  seed={}  epoch={}  test F1={}  test acc={}".format(
        payload.get("seed"), payload.get("epoch"),
        payload.get("test_fscore"), payload.get("test_acc")))

    D_audio = 1582 if saved_args["Dataset"] == "IEMOCAP" else 300
    D_visual, D_text = 342, 1024
    D_g = 512 if saved_args["Dataset"] == "IEMOCAP" else 1024
    n_classes = 6 if saved_args["Dataset"] == "IEMOCAP" else 7
    n_speakers = 2 if saved_args["Dataset"] == "IEMOCAP" else 9

    reloaded = Model(
        saved_args["base_model"], D_text, D_g, 150, 100, 100, 100, 512,
        n_speakers=n_speakers, max_seq_len=200,
        window_past=saved_args["windowp"], window_future=saved_args["windowf"],
        n_classes=n_classes, dropout=saved_args["dropout"],
        no_cuda=saved_args["no_cuda"], graph_type=saved_args["graph_type"],
        use_residue=saved_args["use_residue"], D_m_v=D_visual, D_m_a=D_audio,
        modals=saved_args["modals"], att_type=saved_args["mm_fusion_mthd"],
        dataset=saved_args["Dataset"], use_speaker=saved_args["use_speaker"],
        use_modal=saved_args["use_modal"], norm=saved_args["norm"],
        num_L=saved_args["num_L"], num_K=saved_args["num_K"],
        use_afw=saved_args["use_afw"], afw_rank=saved_args["afw_rank"],
        afw_beta=saved_args["afw_beta"], afw_nlayers=saved_args["afw_nlayers"],
        afw_dropout=saved_args["afw_dropout"],
        pmr_tap=(saved_args["pmr_tap"]
                 if (saved_args["use_pmr"] or saved_args["pmr_probe"]) else None),
        pmr_av_encoder=saved_args["pmr_av_encoder"],
    )
    reloaded.load_state_dict(payload["state_dict"], strict=True)
    reloaded.eval()
    print("  state_dict loaded into a freshly constructed Model: OK")
    print("  parameters: {:,}".format(sum(p.numel() for p in reloaded.parameters())))
    if "pmr_bank" in payload:
        shapes = {k: tuple(v.shape) for k, v in payload["pmr_bank"].items()
                  if k.startswith("proto_")}
        print("  PMR prototype buffers restored:", shapes)

    print("\nTo re-score this checkpoint without retraining, run train.py with")
    print("--testing plus the same module flags it was trained with; it reloads")
    print("the weights and reproduces the saved test F1 exactly:")
    flags = " ".join(
        ["--base-model", str(saved_args["base_model"]),
         "--Dataset", str(saved_args["Dataset"]),
         "--graph_type", str(saved_args["graph_type"]),
         "--mm_fusion_mthd", str(saved_args["mm_fusion_mthd"]),
         "--modals", str(saved_args["modals"]),
         "--norm", str(saved_args["norm"]),
         "--num_L", str(saved_args["num_L"]), "--num_K", str(saved_args["num_K"]),
         "--dropout", str(saved_args["dropout"]), "--multi_modal"]
        + (["--use_afw"] if saved_args["use_afw"] else [])
        + (["--use_amw"] if saved_args["use_amw"] else [])
        + (["--use_pmr"] if saved_args["use_pmr"] else []))
    print("  python train.py --testing --save_path {} {}".format(
        best_path.relative_to(WORK_REPO), flags))


## 8. Outputs

In [ ]:
# =========================== 7. Collect outputs ============================
from pathlib import Path

WORK_REPO = Path("/kaggle/working/M3NET")

patterns = [
    "record_*.pk", "*.pth", "*.pth.tar",
    "checkpoints/*.pth.tar",
    "results_*.csv", "diagnostics*.csv", "seed_sweep_results.csv",
    "log_*.txt", "ablation_results_*.txt",
]

output_files = sorted({p for pattern in patterns for p in WORK_REPO.glob(pattern)})

total_mib = 0.0
print("Generated output files:")
for path in output_files:
    size = path.stat().st_size / 1024 ** 2
    total_mib += size
    print("  {}  ({:.2f} MiB)".format(path.relative_to(WORK_REPO), size))
print("  {:>7.1f} MiB total".format(total_mib))

for path in sorted(WORK_REPO.glob("results_*.csv")):
    print("\n--- {} ---".format(path.name))
    print(path.read_text())

print("\nWhen the notebook finishes, use Save Version so Kaggle preserves files "
      "under /kaggle/working -- including the checkpoints/ directory.")
print("Kaggle caps notebook output at 20 GB; if a large ablation approaches that, "
      "set SAVE_MODEL = False for the screening rows and keep checkpoints only "
      "for the configurations you intend to reuse.")
